# Estimating Annual Growth of Airbnb Hosts




Estimate the growth of Airbnb each year using the number of hosts registered as the growth metric. The rate of growth is calculated by taking ((number of hosts registered in the current year - number of hosts registered in the previous year) / the number of hosts registered in the previous year) * 100. Output the year, number of hosts in the current year, number of hosts in the previous year, and the rate of growth. Round the rate of growth to the nearest percent and order the result in the ascending order based on the year. 

Assume that the dataset consists only of unique hosts, meaning there are no duplicate hosts listed.
This notebook analyzes the yearly growth of Airbnb by calculating the rate of increase in registered hosts. The growth rate is determined using the formula:

\[
\text{Growth Rate (\%)} = \frac{\text{Hosts}_{\text{current year}} - \text{Hosts}_{\text{previous year}}}{\text{Hosts}_{\text{previous year}}} \times 100
\]

We will output the year, number of hosts in the current year, number of hosts in the previous year, and the rounded rate of growth, ordered by year.

🌀 Definitely you are going to enjoy by solving this, you'll learn how to use multiple CTE and windows functions. Give it a try and share the output! 👇

In [0]:
CREATE TABLE ska_catalog.bronze.airbnb_search_details (
  id INT,
  price FLOAT,
  property_type STRING,
  room_type STRING,
  amenities STRING,
  accommodates INT,
  bathrooms INT,
  bed_type STRING,
  cancellation_policy STRING,
  cleaning_fee BOOLEAN,
  city STRING,
  host_identity_verified STRING,
  host_response_rate STRING,
  host_since TIMESTAMP,
  neighbourhood STRING,
  number_of_reviews INT,
  review_scores_rating FLOAT,
  zipcode STRING,
  bedrooms INT,
  beds INT
);

In [0]:

INSERT INTO ska_catalog.bronze.airbnb_search_details (id, price, property_type, room_type, amenities, accommodates, bathrooms, bed_type, cancellation_policy, cleaning_fee, city, host_identity_verified, host_response_rate, host_since, neighbourhood, number_of_reviews, review_scores_rating, zipcode, bedrooms, beds)
VALUES
(7, 150, 'House', 'Entire home/apt', 'WiFi, Kitchen', 5, 2, 'Queen Bed', 'Flexible', TRUE, 'Seattle', 'Yes', '90%', CAST('2019-05-30' AS TIMESTAMP), 'Capitol Hill', 200, 4.6, '98102', 2, 3),
(8, 60, 'Apartment', 'Shared room', 'WiFi', 1, 1, 'Single Bed', 'Moderate', FALSE, 'Boston', 'Yes', '80%', CAST('2018-04-18' AS TIMESTAMP), 'Beacon Hill', 50, 4.2, '02108', 1, 1),
(9, 90, 'House', 'Private room', 'WiFi, Parking', 3, 2, 'King Bed', 'Strict', TRUE, 'Denver', 'No', '85%', CAST('2021-02-10' AS TIMESTAMP), 'Downtown', 75, 4.0, '80202', 1, 2),
(10, 250, 'Villa', 'Entire home/apt', 'Pool, WiFi, Kitchen', 8, 4, 'King Bed', 'Flexible', TRUE, 'Las Vegas', 'Yes', '95%', CAST('2022-06-15' AS TIMESTAMP), 'The Strip', 400, 4.9, '89109', 4, 5);

In [0]:
WITH yearly_host AS(
  SELECT 
    YEAR(host_since) As YEAR,
    COUNT(DISTINCT id) AS `host_in_current_year`
  FROM ska_catalog.bronze.airbnb_search_details
  WHERE host_since IS NOT NULL
  GROUP BY YEAR(host_since)
)

  SELECT
    a.year AS current_year,
    a.host_in_current_year,
    b.host_in_current_year AS `host_in_previous_year`,
    ROUND(
      CASE
        WHEN b.host_in_current_year = 0 THEN NULL
        ELSE (
          (a.host_in_current_year - b.host_in_current_year) * 100.0
        )/ b.host_in_current_year
      END ,0
    ) AS growth_rate
  FROM yearly_host a
  LEFT JOIN yearly_host b
    ON a.year = b.year + 1

In [0]:
WITH yearly_host AS(
  SELECT 
    YEAR(host_since) As YEAR,
    COUNT(DISTINCT id) AS `host_in_current_year`
  FROM ska_catalog.bronze.airbnb_search_details
  GROUP BY YEAR(host_since)
),
GroupCalculation As (
  SELECT
    a.year AS current_year,
    a.host_in_current_year,
    b.host_in_current_year AS `host_in_previous_year`,
    ROUND(
      CASE
        WHEN b.host_in_current_year = 0 THEN NULL
        ELSE (
          (a.host_in_current_year - b.host_in_current_year) * 100.0
        )/ b.host_in_current_year
      END ,0
    ) AS growth_rate
  FROM yearly_host a
  LEFT JOIN yearly_host b
    ON a.year = b.year + 1
)
SELECT 
  current_year,
  host_in_current_year,
  host_in_previous_year,
  growth_rate
FROM
  GroupCalculation
ORDER BY 
  current_year ASc;